# ĐỒ ÁN BIG DATA: HỆ KHUYẾN NGHỊ SẢN PHẨM (RETAILROCKET DATASET)
## NOTEBOOK 02: THỬ NGHIỆM VÀ ĐÁNH GIÁ MÔ HÌNH LỌC CỘNG TÁC ALS
**Thành viên thực hiện:** HUY (Kỹ sư Học máy - Machine Learning Engineer)

### Nội dung thực hiện:
1. Nạp ma trận tương tác User-Item từ tệp Parquet.
2. Khởi tạo phiên làm việc Apache Spark và phân chia Train / Test (80/20).
3. Xây dựng mô hình ALS với cơ chế Implicit Feedback (`implicitPrefs=True`).
4. Đánh giá sai số dự đoán qua thước đo RMSE và MAE.
5. Tinh chỉnh siêu tham số (Hyperparameter Tuning) và so sánh hiệu năng.
6. Trích xuất gợi ý sản phẩm cá nhân hóa Top-K.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

# Khởi tạo Spark Session
spark = SparkSession.builder \
    .appName('Notebook_Test_ALS_Model') \
    .config('spark.master', 'local[*]') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print('Spark Version:', spark.version)

### 1. Nạp ma trận tương tác User-Item 3 cột chuẩn

In [ ]:
matrix_path = '../data/processed/user_item_matrix.parquet'
df_matrix = spark.read.parquet(matrix_path)
print(f'Tổng số cặp tương tác: {df_matrix.count():,}')
df_matrix.show(5)

### 2. Phân chia tập dữ liệu Huấn luyện / Kiểm thử (80% Train, 20% Test)

In [ ]:
train_df, test_df = df_matrix.randomSplit([0.8, 0.2], seed=42)
train_df.cache()
test_df.cache()
print(f'Train records: {train_df.count():,}')
print(f'Test records: {test_df.count():,}')

### 3. Huấn luyện mô hình ALS Implicit Feedback

In [ ]:
# Cấu hình Best Model đã tìm thấy qua Grid Search
als = ALS(
    userCol='userId',
    itemCol='itemId',
    ratingCol='rating',
    implicitPrefs=True,
    coldStartStrategy='drop',
    rank=10,
    regParam=0.10,
    alpha=20.0,
    maxIter=10,
    seed=42
)

print('Đang huấn luyện mô hình ALS...')
model = als.fit(train_df)
print('Huấn luyện hoàn tất!')

### 4. Đánh giá mô hình trên tập kiểm thử (RMSE & MAE)

In [ ]:
predictions = model.transform(test_df)

evaluator_rmse = RegressionEvaluator(metricName='rmse', labelCol='rating', predictionCol='prediction')
evaluator_mae = RegressionEvaluator(metricName='mae', labelCol='rating', predictionCol='prediction')

rmse = evaluator_rmse.evaluate(predictions)
mae = evaluator_mae.evaluate(predictions)

print(f'-> Đánh giá Test RMSE: {rmse:.4f}')
print(f'-> Đánh giá Test MAE:  {mae:.4f}')
predictions.show(5)

### 5. Sinh gợi ý Top-K sản phẩm cho người dùng mẫu

In [ ]:
sample_users = test_df.select('userId').distinct().limit(5)
recommendations = model.recommendForUserSubset(sample_users, 5)
recommendations.show(truncate=False)